# GRPO Training for Methanol APC Environment

**Goal**: Train an LLM to act as an autonomous Advanced Process Control (APC) operator for a methanol synthesis reactor using Group Relative Policy Optimization (GRPO).

**Architecture**: `LLM generates JSON action` → `MethanolRewardFunction parses it` → `env.step()` → `reward back to GRPO`

**Environment**: [HF Space](https://huggingface.co/spaces/glitchfilter/methanol-apc-env) | [GitHub](https://github.com/Bhavneet1492/openenv-methanol-apc)

---
## 1. Setup and Dependencies

In [1]:
%%capture
# Install dependencies (run once on Colab)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl>=0.15" "peft" "accelerate" "bitsandbytes"
!pip install "openenv-core[core]>=0.2.2" numpy matplotlib

# Clone the methanol APC repo for environment code
!git clone https://github.com/Bhavneet1492/openenv-methanol-apc.git /content/methanol-apc 2>/dev/null || true
import sys
sys.path.insert(0, "/content/methanol-apc")

In [2]:
import json, os, random
import numpy as np
import matplotlib.pyplot as plt
import torch
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

# Verify GPU
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "")

NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

## 2. Configuration and Constants

We use the environment's built-in `MethanolGRPOConfig` for recommended hyperparameters, then customize for our Colab GPU budget.

In [ ]:
# === Model Configuration ===
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
LORA_R = 16
LORA_ALPHA = 32
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"]

# === Training Configuration ===
NUM_TRAIN_STEPS = 150        # Enough to show improvement curve
GROUP_SIZE = 4               # GRPO group size (compare 4 completions)
BATCH_SIZE = 2               # Per-device batch size
GRAD_ACCUM = 4               # Effective batch = 2 * 4 = 8
LEARNING_RATE = 5e-6         # Conservative for RL
MAX_COMPLETION_LENGTH = 200  # JSON actions are short (~100 tokens)
TEMPERATURE = 0.7            # Exploration during generation

# === Environment Configuration ===
TASKS = ["optimization", "startup", "disturbance_rejection"]
STEPS_PER_EPISODE = 20       # Short episodes for fast training signal
NUM_PROMPTS = 200            # Total prompt dataset size

# === Paths ===
PLOT_DIR = "/content/methanol-apc/training_plots"
os.makedirs(PLOT_DIR, exist_ok=True)

# === System Prompt (shared across all tasks) ===
SYSTEM_PROMPT = """You are an AI controller for a methanol synthesis reactor (ICI Low-Pressure Process).
Given sensor readings, output a JSON control action with these fields:
{"feed_rate_h2": <0-10>, "feed_rate_co": <0-5>, "cooling_water_flow": <0-100>, "compressor_power": <0-100>}

PHYSICS RULES:
- CO + 2H2 -> CH3OH is exothermic (-90.5 kJ/mol). More feed = more heat + methanol.
- Optimal temperature: 240-260C. Above 270C = catalyst damage. Above 300C = EMERGENCY SHUTDOWN.
- Ideal H2/CO ratio = 2.0. Cooling water removes heat. Higher compressor = higher pressure = faster reaction.
- Revenue: methanol_kg * $0.74/kg. Costs: feed + electricity + cooling.

Respond with ONLY the JSON object. No explanation."""

print(f"Config: {MODEL_NAME}, {NUM_TRAIN_STEPS} steps, group_size={GROUP_SIZE}")

## 3. Helper Functions: Environment, Reward, and Prompts

In [ ]:
# --- Environment wrapper (runs locally, no network needed) ---
from methanol_apc_env.server.methanol_environment import MethanolAPCEnvironment
from methanol_apc_env.models import MethanolAPCAction

def make_env(task="optimization", seed=42):
    """Create a fresh environment instance for a given task."""
    env = MethanolAPCEnvironment()
    obs = env.reset(task_name=task, seed=seed)
    return env, obs

def obs_to_text(obs):
    """Convert observation to a compact sensor-reading string for the LLM prompt."""
    return (
        f"T={obs.temperature:.1f}C P={obs.pressure:.1f}bar "
        f"H2={obs.feed_rate_h2:.2f} CO={obs.feed_rate_co:.2f} ratio={obs.h2_co_ratio:.2f} "
        f"cool={obs.cooling_water_flow:.0f}L/min cat={obs.catalyst_health:.2%} "
        f"rate={obs.reaction_rate:.4f} MeOH={obs.methanol_produced:.1f}kg "
        f"profit={obs.profit_this_step:.3f} total={obs.cumulative_profit:.2f} "
        f"step={obs.step_number}/{obs.max_steps} task={obs.task_name}"
    )

# --- Reward function (wraps the environment physics) ---
def reward_fn(completions: list[str], **kwargs) -> list[float]:
    """
    Score a batch of LLM completions by stepping the environment.
    Each completion should be a JSON action string.
    Returns list of rewards in (0.01, 0.99).
    """
    rewards = []
    for completion in completions:
        try:
            # Extract JSON from completion (handle markdown fences)
            text = completion.strip()
            if "```" in text:
                text = text.split("```")[1] if "```" in text else text
                text = text.replace("json", "").strip()
            action_dict = json.loads(text)
            action = MethanolAPCAction(**action_dict)
            env, _ = make_env(task=random.choice(TASKS), seed=random.randint(0, 9999))
            obs = env.step(action)
            rewards.append(float(obs.reward))
        except Exception:
            rewards.append(0.01)  # Minimum reward for invalid output
    return rewards

# --- Build prompt dataset from environment observations ---
def build_prompt_dataset(num_prompts=NUM_PROMPTS):
    """Generate prompts by running the env with random actions and capturing observations."""
    prompts = []
    for i in range(num_prompts):
        task = random.choice(TASKS)
        env, obs = make_env(task=task, seed=i)
        # Take a few random steps to get varied observations
        for _ in range(random.randint(0, 5)):
            action = MethanolAPCAction(
                feed_rate_h2=random.uniform(1, 8),
                feed_rate_co=random.uniform(0.5, 4),
                cooling_water_flow=random.uniform(10, 80),
                compressor_power=random.uniform(30, 80),
            )
            obs = env.step(action)
            if obs.done:
                break

        sensor_text = obs_to_text(obs)
        prompt = f"{SYSTEM_PROMPT}\n\nSensors:\n{sensor_text}\n\nAction JSON:"
        prompts.append({"prompt": prompt})
    return Dataset.from_list(prompts)

# Test: generate one prompt and one reward
env, obs = make_env("optimization")
test_prompt = obs_to_text(obs)
test_reward = reward_fn(['{"feed_rate_h2": 5, "feed_rate_co": 2.5, "cooling_water_flow": 50, "compressor_power": 60}'])
print(f"Sample sensor text:\n{test_prompt}\n")
print(f"Test reward for reasonable action: {test_reward[0]:.4f}")
print(f"Test reward for garbage input: {reward_fn(['not json'])[0]:.4f}")

## 4. Load Model, Build Dataset, and Train with GRPO

This is the core training loop. We:
1. Load Qwen2.5-7B with Unsloth 4-bit quantization + LoRA
2. Build the prompt dataset from environment observations
3. Run GRPO training with our physics-based reward function

In [ ]:
# --- Load Model with Unsloth ---
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-detect
    load_in_4bit=True,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"LoRA: r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGET_MODULES}")

In [ ]:
# --- Build Prompt Dataset ---
print("Building prompt dataset from environment observations...")
dataset = build_prompt_dataset(NUM_PROMPTS)
print(f"Dataset size: {len(dataset)} prompts")
print(f"\nSample prompt (truncated):\n{dataset[0]['prompt'][:300]}...")

In [ ]:
# --- Configure and Run GRPO Training ---
training_args = GRPOConfig(
    output_dir="./grpo_methanol_output",
    max_steps=NUM_TRAIN_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    max_completion_length=MAX_COMPLETION_LENGTH,
    num_generations=GROUP_SIZE,
    temperature=TEMPERATURE,
    logging_steps=5,
    save_steps=50,
    report_to="none",  # We save plots manually
    bf16=True,
    seed=42,
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    reward_funcs=reward_fn,
    tokenizer=tokenizer,
)

print("Starting GRPO training...")
print(f"  Steps: {NUM_TRAIN_STEPS}, Batch: {BATCH_SIZE}x{GRAD_ACCUM}, LR: {LEARNING_RATE}")
print(f"  Group size: {GROUP_SIZE}, Max completion: {MAX_COMPLETION_LENGTH} tokens")

train_result = trainer.train()
print(f"\nTraining complete! Final loss: {train_result.training_loss:.4f}")

## 5. Generate Training Plots and Evaluate

Save loss curve, reward curve, and baseline-vs-trained comparison as `.png` files committed to the repo.

In [ ]:
# --- Extract training logs ---
log_history = trainer.state.log_history
steps = [e["step"] for e in log_history if "loss" in e]
losses = [e["loss"] for e in log_history if "loss" in e]

# --- Plot 1: Loss Curve ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(steps, losses, color="#3b82f6", linewidth=2, label="Training Loss")
ax.set_xlabel("Training Step", fontsize=12)
ax.set_ylabel("Loss", fontsize=12)
ax.set_title("GRPO Training Loss — Methanol APC Environment", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f"{PLOT_DIR}/loss_curve.png", dpi=150, bbox_inches="tight")
print(f"Saved: {PLOT_DIR}/loss_curve.png")
plt.show()

In [ ]:
# --- Evaluate: Untrained Baseline vs Trained Agent ---
def evaluate_agent(model, tokenizer, task="optimization", num_episodes=5, steps_per_ep=15, label="Agent"):
    """Run the model against the environment and collect per-step rewards."""
    FastLanguageModel.for_inference(model)
    all_rewards = []
    for ep in range(num_episodes):
        env, obs = make_env(task=task, seed=ep * 100)
        ep_rewards = []
        for step in range(steps_per_ep):
            if obs.done:
                break
            sensor_text = obs_to_text(obs)
            prompt = f"{SYSTEM_PROMPT}\n\nSensors:\n{sensor_text}\n\nAction JSON:"
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, max_new_tokens=150, temperature=0.3,
                    do_sample=True, pad_token_id=tokenizer.eos_token_id
                )
            response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            try:
                action_dict = json.loads(response.strip().split("\n")[0])
                action = MethanolAPCAction(**action_dict)
                obs = env.step(action)
                ep_rewards.append(float(obs.reward))
            except Exception:
                # Fallback: safe conservative action
                obs = env.step(MethanolAPCAction(
                    feed_rate_h2=3.0, feed_rate_co=1.5,
                    cooling_water_flow=60.0, compressor_power=50.0))
                ep_rewards.append(float(obs.reward))
        all_rewards.append(ep_rewards)
    # Pad to same length and average
    max_len = max(len(r) for r in all_rewards)
    padded = [r + [r[-1]] * (max_len - len(r)) if r else [0.01] * max_len for r in all_rewards]
    avg_rewards = np.mean(padded, axis=0)
    print(f"{label} [{task}]: avg_reward={np.mean(avg_rewards):.4f}, episodes={num_episodes}")
    return avg_rewards

# --- Run baseline (untrained = random/fallback actions) ---
def evaluate_baseline(task="optimization", num_episodes=5, steps_per_ep=15):
    """Baseline: random actions from the environment's action space."""
    all_rewards = []
    for ep in range(num_episodes):
        env, obs = make_env(task=task, seed=ep * 100)
        ep_rewards = []
        for step in range(steps_per_ep):
            if obs.done:
                break
            action = MethanolAPCAction(
                feed_rate_h2=random.uniform(1, 8),
                feed_rate_co=random.uniform(0.5, 4),
                cooling_water_flow=random.uniform(10, 80),
                compressor_power=random.uniform(20, 80),
            )
            obs = env.step(action)
            ep_rewards.append(float(obs.reward))
        all_rewards.append(ep_rewards)
    max_len = max(len(r) for r in all_rewards)
    padded = [r + [r[-1]] * (max_len - len(r)) if r else [0.01] * max_len for r in all_rewards]
    avg_rewards = np.mean(padded, axis=0)
    print(f"Baseline [{task}]: avg_reward={np.mean(avg_rewards):.4f}")
    return avg_rewards

print("Evaluating baseline (random actions)...")
baseline_rewards = evaluate_baseline("optimization")
print("\nEvaluating trained agent...")
trained_rewards = evaluate_agent(model, tokenizer, "optimization", label="Trained")

In [ ]:
# --- Plot 2: Reward Curve (average reward per step during eval) ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(len(trained_rewards)), trained_rewards, color="#10b981", linewidth=2, label="Trained Agent")
ax.axhline(y=np.mean(trained_rewards), color="#10b981", linestyle="--", alpha=0.5, label=f"Trained Mean: {np.mean(trained_rewards):.3f}")
ax.set_xlabel("Environment Step", fontsize=12)
ax.set_ylabel("Reward (per step)", fontsize=12)
ax.set_title("GRPO Trained Agent — Reward per Step (Optimization Task)", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f"{PLOT_DIR}/reward_curve.png", dpi=150, bbox_inches="tight")
print(f"Saved: {PLOT_DIR}/reward_curve.png")
plt.show()

# --- Plot 3: Baseline vs Trained (same axes) ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(len(baseline_rewards)), baseline_rewards, color="#ef4444", linewidth=2, alpha=0.8, label=f"Random Baseline (avg: {np.mean(baseline_rewards):.3f})")
ax.plot(range(len(trained_rewards)), trained_rewards, color="#10b981", linewidth=2, label=f"GRPO Trained (avg: {np.mean(trained_rewards):.3f})")
ax.fill_between(range(len(baseline_rewards)), baseline_rewards, alpha=0.1, color="#ef4444")
ax.fill_between(range(len(trained_rewards)), trained_rewards, alpha=0.1, color="#10b981")
ax.set_xlabel("Environment Step", fontsize=12)
ax.set_ylabel("Reward (per step)", fontsize=12)
ax.set_title("Baseline vs GRPO-Trained Agent — Methanol APC", fontsize=14)
ax.legend(fontsize=11, loc="lower right")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f"{PLOT_DIR}/baseline_vs_trained.png", dpi=150, bbox_inches="tight")
print(f"Saved: {PLOT_DIR}/baseline_vs_trained.png")
plt.show()

print(f"\n{'='*50}")
print(f"IMPROVEMENT: {np.mean(trained_rewards) - np.mean(baseline_rewards):+.4f} reward")
print(f"Baseline avg: {np.mean(baseline_rewards):.4f}")
print(f"Trained avg:  {np.mean(trained_rewards):.4f}")
print(f"{'='*50}")

In [ ]:
# --- Save model (optional: push to HF Hub) ---
model.save_pretrained("./grpo_methanol_trained")
tokenizer.save_pretrained("./grpo_methanol_trained")
print("Model saved to ./grpo_methanol_trained")

# Copy plots to repo (if running from Colab with cloned repo)
import shutil
for fname in ["loss_curve.png", "reward_curve.png", "baseline_vs_trained.png"]:
    src = f"{PLOT_DIR}/{fname}"
    if os.path.exists(src):
        print(f"Plot ready: {src}")

print("\n--- DONE ---")
print("Next steps:")
print("1. Download plots from training_plots/ and commit to repo")
print("2. Update README.md with embedded plot images")
print("3. Push to GitHub")
print("4. Publish blog post on HuggingFace")